In [1]:
from dotenv import load_dotenv
load_dotenv()

import kaggle_benchmarks as kbench
import pandas as pd
import numpy as np
import re
import traceback

# ==========================================
# Judge Implementation
# ==========================================

def extract_code(llm_response: str) -> str:
    """Extracts python code from the LLM response.
    Handles triple backtick blocks (```python ... ```) or returns the raw response if no blocks are found.
    """
    pattern = r"```(?:python)?\s*(.*?)\s*```"
    match = re.search(pattern, llm_response, re.DOTALL)
    if match:
        return match.group(1).strip()
    return llm_response.strip()

def run_code(code: str, df: pd.DataFrame) -> tuple[any, pd.DataFrame]:
    """Executes the given code snippet on a copy of df and returns the evaluation output and the final df.
    """
    local_vars = {'pd': pd, 'df': df.copy()}
    code_stripped = code.strip()
    
    # Pre-process code to handle Pandas 3.0+ Copy-on-Write chained inplace operations
    pattern = r"df\[['\"](\w+)['\"]\]\.fillna\((.*),\s*inplace\s*=\s*True\)"
    if re.search(pattern, code_stripped):
        code_stripped = re.sub(pattern, r"df['\1'] = df['\1'].fillna(\2)", code_stripped)
    
    # Try eval first in case it's a pure expression (e.g. df.duplicated().sum() or df['Dept'].value_counts())
    try:
        val = eval(code_stripped, {}, local_vars)
        return val, local_vars['df']
    except Exception:
        # If eval fails, it's likely a statement (e.g. assignment or inplace call)
        # We run it with exec.
        exec(code_stripped, {}, local_vars)
        # Check if 'result' is defined in case they assigned a scalar to result
        # e.g., result = df.duplicated().sum()
        if 'result' in local_vars:
            return local_vars['result'], local_vars['df']
        return None, local_vars['df']

def is_equivalent(val1: any, val2: any) -> bool:
    """Helper to check if two variables (DataFrames, Series, Indexes, or scalars) are equivalent."""
    if isinstance(val1, pd.DataFrame) and isinstance(val2, pd.DataFrame):
        try:
            pd.testing.assert_frame_equal(val1, val2, check_dtype=False, check_index_type=False)
            return True
        except AssertionError:
            return False
    elif isinstance(val1, pd.Series) and isinstance(val2, pd.Series):
        try:
            pd.testing.assert_series_equal(val1, val2, check_dtype=False, check_index_type=False)
            return True
        except AssertionError:
            return False
    elif isinstance(val1, pd.Index) and isinstance(val2, pd.Index):
        try:
            pd.testing.assert_index_equal(val1, val2, check_exact=False)
            return True
        except AssertionError:
            return False
    else:
        try:
            if pd.isna(val1) and pd.isna(val2):
                return True
            return val1 == val2
        except Exception:
            return False

def evaluate_code(code: str, df: pd.DataFrame, expected_code: str) -> bool:
    """Executes the given model code and expected code against the DataFrame and checks if the outputs are equivalent.
    """
    try:
        # 1. Run the expected reference code to get expected results
        expected_val, expected_df_after = run_code(expected_code, df)
        
        # 2. Run the model's code to get actual results
        model_val, model_df_after = run_code(code, df)
        
        # 3. Check if expected result was a scalar / non-DataFrame value
        if expected_val is not None and not isinstance(expected_val, (pd.DataFrame, pd.Series, pd.Index)):
            # If the model code returns the expected value (directly or via variable assignment)
            if is_equivalent(model_val, expected_val):
                return True
            return False
            
        # 4. If the expected result is a DataFrame/Series/Index (either returned or modified df)
        # Determine the target expected output
        if isinstance(expected_val, (pd.DataFrame, pd.Series, pd.Index)):
            target_expected = expected_val
        else:
            target_expected = expected_df_after
            
        # Check if the model's evaluated return value matches target_expected
        if isinstance(model_val, (pd.DataFrame, pd.Series, pd.Index)) and is_equivalent(model_val, target_expected):
            return True
            
        # Check if the model's modified DataFrame matches target_expected
        if is_equivalent(model_df_after, target_expected):
            return True
            
        return False
        
    except Exception as e:
        print(f"Error evaluating model code: {e}")
        traceback.print_exc()
        return False

# ==========================================
# Test Cases Data Definitions
# ==========================================

def get_test_cases():
    """Returns the comprehensive list of 12 test cases for the 6 EDA categories."""
    return [
        # --- Category 1: Missing Values ---
        {
            'id': 'missing_values_median',
            'category': 'Missing Values',
            'name': 'Fill missing numerical column with the median',
            'get_df': lambda: pd.DataFrame({'col': [1.0, 2.0, None, 4.0, 10.0]}),
            'prompt': (
                "You are given a pandas DataFrame named 'df'. The column 'col' is numerical and contains missing values (NaN). "
                "Write the Python Pandas code to fill these missing values in 'col' with the median value of the 'col' column. "
                "Modify the DataFrame in-place or assign the result back to df['col']. "
                "Provide only the code snippet to achieve this."
            ),
            'expected_code': "df['col'].fillna(df['col'].median(), inplace=True)"
        },
        {
            'id': 'missing_values_dropna',
            'category': 'Missing Values',
            'name': 'Drop rows where ANY value is missing',
            'get_df': lambda: pd.DataFrame({
                'A': [1.0, None, 3.0],
                'B': [4.0, 5.0, None],
                'C': [7.0, 8.0, 9.0]
            }),
            'prompt': (
                "You are given a pandas DataFrame named 'df'. Write the Python Pandas code to drop all rows where ANY value is missing. "
                "Modify the DataFrame in-place or assign the result back to 'df'. "
                "Provide only the code snippet to achieve this."
            ),
            'expected_code': "df.dropna(inplace=True)"
        },
        # --- Category 2: Duplicates ---
        {
            'id': 'duplicates_drop',
            'category': 'Duplicates',
            'name': 'Remove duplicate rows but keep the first occurrence',
            'get_df': lambda: pd.DataFrame({
                'A': [1, 2, 1, 2, 3],
                'B': [4, 5, 4, 5, 6]
            }),
            'prompt': (
                "You are given a pandas DataFrame named 'df' containing duplicate rows. "
                "Write the Python Pandas code to remove duplicate rows but keep the first occurrence of each unique row. "
                "Modify the DataFrame in-place or assign the result back to 'df'. "
                "Provide only the code snippet to achieve this."
            ),
            'expected_code': "df.drop_duplicates(inplace=True)"
        },
        {
            'id': 'duplicates_count',
            'category': 'Duplicates',
            'name': 'Count how many duplicate rows exist',
            'get_df': lambda: pd.DataFrame({
                'A': [1, 2, 1, 2, 3],
                'B': [4, 5, 4, 5, 6]
            }),
            'prompt': (
                "You are given a pandas DataFrame named 'df'. Write the Python Pandas code or expression to count how many duplicate rows exist in the DataFrame. "
                "Provide only the code snippet or expression to achieve this."
            ),
            'expected_code': "df.duplicated().sum()"
        },
        # --- Category 3: Data Types & Casting ---
        {
            'id': 'casting_price_float',
            'category': 'Data Types & Casting',
            'name': "Convert a column named 'Price' from string/object type to float",
            'get_df': lambda: pd.DataFrame({'Price': ['10.5', '20.0', '30.25']}),
            'prompt': (
                "You are given a pandas DataFrame named 'df' where the column 'Price' contains string/object representations of floating-point numbers. "
                "Write the Python Pandas code to convert the 'Price' column to float data type. "
                "Provide only the code snippet to achieve this."
            ),
            'expected_code': "df['Price'] = df['Price'].astype(float)"
        },
        {
            'id': 'casting_date_datetime',
            'category': 'Data Types & Casting',
            'name': "Convert a 'Date' column from string to datetime format",
            'get_df': lambda: pd.DataFrame({'Date': ['2026-01-01', '2026-06-09', '2026-12-31']}),
            'prompt': (
                "You are given a pandas DataFrame named 'df' where the 'Date' column contains date strings. "
                "Write the Python Pandas code to convert this column to a datetime format. "
                "Provide only the code snippet to achieve this."
            ),
            'expected_code': "df['Date'] = pd.to_datetime(df['Date'])"
        },
        # --- Category 4: Filtering & Sorting ---
        {
            'id': 'filter_age_country',
            'category': 'Filtering & Sorting',
            'name': "Filter the dataframe to only show rows where 'Age' is greater than 18 AND 'Country' is 'USA'",
            'get_df': lambda: pd.DataFrame({
                'Age': [15, 25, 30, 20],
                'Country': ['USA', 'USA', 'Canada', 'USA']
            }),
            'prompt': (
                "You are given a pandas DataFrame named 'df'. Write the Python Pandas code to filter the DataFrame to only show rows "
                "where 'Age' is greater than 18 AND 'Country' is 'USA'. "
                "Provide only the code snippet or expression."
            ),
            'expected_code': "df[(df['Age'] > 18) & (df['Country'] == 'USA')]"
        },
        {
            'id': 'sort_salary_desc',
            'category': 'Filtering & Sorting',
            'name': "Sort the dataframe by 'Salary' in descending order",
            'get_df': lambda: pd.DataFrame({
                'Employee': ['A', 'B', 'C'],
                'Salary': [50000, 70000, 60000]
            }),
            'prompt': (
                "You are given a pandas DataFrame named 'df'. Write the Python Pandas code to sort the DataFrame by the 'Salary' column in descending order. "
                "Modify the DataFrame in-place or assign the result back to 'df'. "
                "Provide only the code snippet to achieve this."
            ),
            'expected_code': "df.sort_values(by='Salary', ascending=False, inplace=True)"
        },
        # --- Category 5: Grouping & Aggregation ---
        {
            'id': 'group_dept_value_counts',
            'category': 'Grouping & Aggregation',
            'name': "Find the count of unique values in the 'Department' column",
            'get_df': lambda: pd.DataFrame({'Department': ['HR', 'IT', 'IT', 'HR', 'HR', 'Finance']}),
            'prompt': (
                "You are given a pandas DataFrame named 'df'. Write the Python Pandas code or expression to find the counts of unique values in the 'Department' column. "
                "Provide only the code snippet or expression to achieve this."
            ),
            'expected_code': "df['Department'].value_counts()"
        },
        {
            'id': 'group_region_max_revenue',
            'category': 'Grouping & Aggregation',
            'name': "Calculate the maximum 'Revenue' for each 'Region'",
            'get_df': lambda: pd.DataFrame({
                'Region': ['North', 'North', 'South', 'South', 'East'],
                'Revenue': [100, 150, 200, 180, 300]
            }),
            'prompt': (
                "You are given a pandas DataFrame named 'df'. Write the Python Pandas code or expression to calculate the maximum 'Revenue' for each 'Region'. "
                "Provide only the code snippet or expression to achieve this."
            ),
            'expected_code': "df.groupby('Region')['Revenue'].max()"
        },
        # --- Category 6: String Operations ---
        {
            'id': 'string_lower_name',
            'category': 'String Operations',
            'name': "Convert all strings in the 'Name' column to lowercase",
            'get_df': lambda: pd.DataFrame({'Name': ['ALICE', 'Bob', 'charlie']}),
            'prompt': (
                "You are given a pandas DataFrame named 'df'. Write the Python Pandas code to convert all string values in the 'Name' column to lowercase. "
                "Provide only the code snippet to achieve this."
            ),
            'expected_code': "df['Name'] = df['Name'].str.lower()"
        },
        {
            'id': 'string_strip_city',
            'category': 'String Operations',
            'name': "Remove leading and trailing whitespace from the 'City' column",
            'get_df': lambda: pd.DataFrame({'City': [' New York ', '  London', 'Paris  ']}),
            'prompt': (
                "You are given a pandas DataFrame named 'df'. Write the Python Pandas code to remove leading and trailing whitespace from the values in the 'City' column. "
                "Provide only the code snippet to achieve this."
            ),
            'expected_code': "df['City'] = df['City'].str.strip()"
        }
    ]

# ==========================================
# Task Decorator and Execution
# ==========================================

@kbench.task(name="LLM EDA Mastery Test")
def llm_eda_mastery_test(llm):
    test_cases = get_test_cases()
    
    for tc in test_cases:
        tc_id = tc['id']
        category = tc['category']
        name = tc['name']
        prompt = tc['prompt']
        expected_code = tc['expected_code']
        
        # Build fresh mock DataFrame
        df = tc['get_df']()
        
        # Run each test case in its own isolated chat context to prevent leakage
        with kbench.chats.new(tc_id):
            response = llm.prompt(prompt)
            code = extract_code(response)
            is_correct = evaluate_code(code, df, expected_code)
            
            # Assert the outcome using the expected benchmark assertion framework
            kbench.assertions.assert_true(
                is_correct,
                expectation=f"[{category} - {name}] The generated code should correctly solve the prompt."
            )

In [2]:
if __name__ == "__main__":
    llm_eda_mastery_test.run(kbench.llm)